# NYC Elevator Complaints — District Hotspots

> **Narrative:** *"Here are the buildings in your district that have failed their tenants the most in the last 12 months."*

Covers the six priority council districts: Sanchez (D17), Stevens (D16), Farías (D18), Banks (D42), Hudson (D35), De La Rosa (D10).  
Source: [NYC Open Data — DOB Elevator Complaint Dispositions](https://data.cityofnewyork.us/Housing-Development/DOB-Elevator-Complaint-Dispositions/kqwi-7ncn)

In [ ]:
# ── PARAMETERS — edit before presenting ──────────────────────────
YEARS  = [2025, 2026]    # years to include
TOP_N  = 10              # buildings per district
FOCUS  = None            # set to a key below to show one district only
                         # options: "sanchez" "stevens" "farias" "banks" "hudson" "delarosa"
# ─────────────────────────────────────────────────────────────────

import os, requests
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../../../.env"))
except ImportError:
    pass

TOKEN = os.getenv("SODA_APP_TOKEN")
if not TOKEN:
    print("⚠  SODA_APP_TOKEN not set")

SODA_URL    = "https://data.cityofnewyork.us/resource/kqwi-7ncn.json"
YEAR_FILTER = "(" + " OR ".join(f"date_entered LIKE '%/{y}'" for y in YEARS) + ")"
YEAR_LABEL  = " + ".join(str(y) for y in sorted(YEARS))

DISTRICTS = [
    {"key": "sanchez",  "name": "Justin Sanchez",    "district": "D17",
     "borough": "Bronx",     "cbs": ["201", "202"],
     "neighborhoods": "Hunts Point · Port Morris · Mott Haven · Longwood"},
    {"key": "stevens",  "name": "Althea Stevens",    "district": "D16",
     "borough": "Bronx",     "cbs": ["203"],
     "neighborhoods": "Morrisania · Crotona · Melrose"},
    {"key": "farias",   "name": "Amanda Farías",     "district": "D18",
     "borough": "Bronx",     "cbs": ["209"],
     "neighborhoods": "Parkchester · Castle Hill · Soundview"},
    {"key": "banks",    "name": "Chris Banks",       "district": "D42",
     "borough": "Brooklyn",  "cbs": ["316", "317"],
     "neighborhoods": "Brownsville · East New York"},
    {"key": "hudson",   "name": "Crystal Hudson",    "district": "D35",
     "borough": "Brooklyn",  "cbs": ["308", "309"],
     "neighborhoods": "Crown Heights · Prospect Heights"},
    {"key": "delarosa", "name": "Carmen De La Rosa", "district": "D10",
     "borough": "Manhattan", "cbs": ["112"],
     "neighborhoods": "Washington Heights · Inwood"},
]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f5f5f5",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.family":      "sans-serif",
    "axes.titlesize":   13,
})

if FOCUS:
    districts_to_show = [d for d in DISTRICTS if d["key"] == FOCUS]
else:
    districts_to_show = DISTRICTS

print(f"Parameters: years={YEAR_LABEL}  top_n={TOP_N}  focus={FOCUS or 'all'}")

In [ ]:
def fetch_district(cbs: list, year_filter: str, top_n: int) -> list:
    cb_list = ", ".join(f"'{cb}'" for cb in cbs)
    params = {
        "$select": "house_number, house_street, community_board, count(*) AS complaint_count",
        "$where":  f"complaint_category IN ('6S','6M') AND community_board IN ({cb_list}) AND {year_filter}",
        "$group":  "house_number, house_street, community_board",
        "$order":  "complaint_count DESC",
        "$limit":  str(top_n),
    }
    if TOKEN:
        params["$$app_token"] = TOKEN
    r = requests.get(SODA_URL, params=params, timeout=15)
    r.raise_for_status()
    return r.json()

print(f"Fetching hotspots for {len(districts_to_show)} district(s)...")
results = {}
for d in districts_to_show:
    results[d["key"]] = fetch_district(d["cbs"], YEAR_FILTER, TOP_N)
    print(f"  ✓ {d['name']} ({d['district']}) — {len(results[d['key']])} buildings")

## Hotspot Charts

One chart per district. Use `FOCUS = "sanchez"` in the parameters cell to isolate a single district for a meeting.

In [ ]:
BOROUGH_COLOR = {"Bronx": "#e63946", "Brooklyn": "#e63946", "Manhattan": "#457b9d"}

for d in districts_to_show:
    buildings = results[d["key"]]
    if not buildings:
        print(f"No data for {d['name']}")
        continue

    addrs  = [r.get("house_number", "") + " " + r.get("house_street", "") for r in buildings]
    counts = [int(r.get("complaint_count", 0)) for r in buildings]
    color  = BOROUGH_COLOR.get(d["borough"], "#457b9d")

    fig, ax = plt.subplots(figsize=(9, len(buildings) * 0.42 + 1.6))
    bars = ax.barh(addrs, counts, color=color, height=0.6, edgecolor="white")

    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + max(counts) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                str(count), va="center", fontsize=9, color="#333")

    ax.set_xlabel("Complaints")
    ax.set_title(
        f"{d['name']} ({d['district']}) — Top {len(buildings)} Buildings\n"
        f"{d['neighborhoods']} | {YEAR_LABEL}",
        pad=10, loc="left"
    )
    ax.set_xlim(0, max(counts) * 1.18)
    ax.invert_yaxis()
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()
    plt.show()
    print()

## Side-by-Side Summary

Top building per district at a glance.

In [ ]:
summary = []
for d in DISTRICTS:
    bldgs = results.get(d["key"], [])
    if bldgs:
        top = bldgs[0]
        addr  = top.get("house_number", "") + " " + top.get("house_street", "")
        count = int(top.get("complaint_count", 0))
    else:
        addr, count = "(no data)", 0
    summary.append((f"{d['name']}\n{d['district']} · {d['borough']}", addr, count))

labels  = [s[0] for s in summary]
addrs   = [s[1] for s in summary]
counts  = [s[2] for s in summary]
borough_colors = [
    BOROUGH_COLOR.get(d["borough"], "#457b9d") for d in DISTRICTS
]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, counts, color=borough_colors, edgecolor="white", width=0.55)

for bar, addr, count in zip(bars, addrs, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(counts) * 0.01,
            str(count), ha="center", fontsize=11, fontweight="bold", color="#333")
    ax.text(bar.get_x() + bar.get_width() / 2, -max(counts) * 0.06,
            addr[:22], ha="center", fontsize=7.5, color="#555", rotation=0)

ax.set_ylabel("Complaints (top building)")
ax.set_title(f"Worst Building Per District — {YEAR_LABEL}", pad=12)
ax.set_ylim(-max(counts) * 0.15, max(counts) * 1.18)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
fig.tight_layout()
plt.show()